Databricks notebook source
SENSE 프로젝트 — Silver Layer: 반도체 수출 raw → silver
담당: 1조 (정형 데이터)

목적:
  ADLS Gen2 raw/public_data/semiconductor/ 에서 반도체 수출 데이터를 읽어
  시계열 정제 후 silver/semiconductor/ 에 저장

raw 경로:
  public_data/semiconductor/source=data_go_kr/semiconductor_trade_2024_to_2026.parquet

[데이터 특성]
  - 출처: data.go.kr (관세청 수출입 무역통계 API)
  - 주기: 10일 단위 배치 발표 (일별 데이터 아님)
  - HS Code '8542' (반도체) 필터 적용 필요
  - 주기 불일치(10일 → 일별) Forward Fill 처리 필요

[파생 컬럼]
  export_change_pct : 전기 대비 반도체 수출 변화율
  export_momentum   : 수출 모멘텀 플래그 (전기 대비 -10% 이하 급감 시 1)


# 0. 스토리지 계정 설정 및 ADLS OAuth 인증


In [0]:
# ============================================================
# 스토리지 계정 및 경로 상수 정의
# ============================================================
STORAGE_ACCOUNT  = "3dtteam1adls"
SECRET_SCOPE     = "sense-kv"
BRONZE_CONTAINER    = "raw"
SILVER_CONTAINER = "curated"

BASE_PATH_BRONZE    = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
BASE_PATH_SILVER = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

CALENDAR_PATH = f"{BASE_PATH_SILVER}/master_calendar.parquet"
SEMI_BRONZE_PATH = f"{BASE_PATH_BRONZE}/public_data/semiconductor/source=data_go_kr/semiconductor_trade_2024_to_2026.parquet"
SEMI_SILVER_PATH   = f"{BASE_PATH_SILVER}/semiconductor"

# ============================================================
# ADLS Gen2 OAuth 인증 (Service Principal)
# ============================================================
spark.conf.set(
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net", "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get(scope=SECRET_SCOPE, key="adls-client-id")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get(scope=SECRET_SCOPE, key="adls-client-secret")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{dbutils.secrets.get(scope=SECRET_SCOPE, key='adls-tenant-id')}/oauth2/token"
)
print(f"✅ ADLS OAuth 인증 설정 완료: {STORAGE_ACCOUNT}")


# 1. master_calendar 로드

반도체 수출 데이터는 **10일 주기**로 발표됩니다.
(매월 1~10일, 11~20일, 21~말일 단위)
캘린더를 기준으로 일별로 Forward Fill 해서
Gold JOIN 시 날짜를 맞춥니다.


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, to_date
from pyspark.sql import Window

calendar_df = (
    spark.read.parquet(CALENDAR_PATH)
    .withColumn("기준일자", to_date(col("기준일자"), "yyyy-MM-dd"))
    .select("기준일자", "한국_휴장일_여부", "미국_휴장일_여부")
)

kr_biz_days = (
    calendar_df
    .filter(col("한국_휴장일_여부") == False)
    .select(col("기준일자").alias("date"))
)

print(f"✅ master_calendar 로드 완료: 한국 영업일 {kr_biz_days.count()}일")


# 2. 반도체 수출 raw 데이터 로드 및 컬럼 확인

> ⚠️ 실제 컬럼명을 먼저 확인 후 아래 정제 셀을 수정하세요.
> 예상 컬럼: `PRD_DE`(기간), `HS_CD`(HS코드), `EXP_USD_AMT`(수출금액)


In [0]:
semi_raw_df = spark.read.parquet(SEMI_BRONZE_PATH)

print(f"✅ 반도체 수출 raw 로드 완료: {semi_raw_df.count()}행")
print(f"\n컬럼 목록 및 타입:")
semi_raw_df.printSchema()
print(f"\n샘플 데이터:")
display(semi_raw_df.limit(10))


# 3. 스키마 정리 및 HS Code 필터

> ⚠️ 위 셀에서 확인한 실제 컬럼명으로 수정 후 실행하세요.

처리 내용:
- 날짜 컬럼 → `date` (DateType)
- HS Code `8542` (반도체 집적회로) 필터
- 수출금액 컬럼 → `export_usd` (double)
- 불필요 컬럼 제거


In [0]:
# ── 실제 컬럼명에 맞게 매핑 ────────────────────────────────────────
DATE_COL   = "year"           # 기간 컬럼 (예: 2024.01 = 2024년 1월)
HS_COL     = "hsCode"         # HS 코드 컬럼 (long)
EXPORT_COL = "expDlr"         # 수출금액 컬럼 (USD)
HS_SEMI    = "8542"           # 반도체 HS Code

semi_clean_df = (
    semi_raw_df
    .withColumn("date",       to_date(F.format_string("%.2f", col(DATE_COL)), "yyyy.MM"))
    .withColumn("export_usd", col(EXPORT_COL).cast("double"))
    .filter(col(HS_COL).cast("string").startswith(HS_SEMI))
    .filter(col("date").isNotNull())
    .filter(col("export_usd").isNotNull())
    .groupBy("date").agg(F.sum("export_usd").alias("export_usd"))
    .orderBy("date")
)

print(f"✅ 스키마 정리 완료: {semi_clean_df.count()}행")
display(semi_clean_df.limit(10))

# 4. 10일 → 일별 Forward Fill (주기 불일치 해소)

반도체 수출은 10일에 한 번 발표됩니다.
yfinance / FRED 는 매 영업일 데이터가 있습니다.
Gold JOIN 을 위해 **캘린더 영업일 기준으로 일별 확장 후 Forward Fill** 합니다.

```
발표 주기 예시:
  2024-01-10: export_usd = 5,200,000  ← 실제 값
  2024-01-11: null                     ← Forward Fill → 5,200,000
  2024-01-12: null                     ← Forward Fill → 5,200,000
  ...
  2024-01-20: export_usd = 4,800,000  ← 실제 값 (다음 발표)
```


In [0]:
# ── semi_clean_df의 매월 1일 → 해당 월 첫 영업일로 매핑 ──────────
first_biz_per_month = (
    kr_biz_days
    .withColumn("ym", F.date_format("date", "yyyy-MM"))
    .groupBy("ym")
    .agg(F.min("date").alias("first_biz_date"))
)

semi_mapped_df = (
    semi_clean_df
    .withColumn("ym", F.date_format("date", "yyyy-MM"))
    .drop("date")
    .join(first_biz_per_month, on="ym", how="inner")
    .withColumnRenamed("first_biz_date", "date")
    .select("date", "export_usd")
)

# ── Forward Fill 전에 파생 컬럼 계산 (월별 원본 기준) ──────────────
w_monthly = Window.orderBy("date")

semi_derived_df = (
    semi_mapped_df
    .withColumn(
        "export_change_pct",
        F.round(
            (col("export_usd") - F.lag("export_usd", 1).over(w_monthly))
            / F.lag("export_usd", 1).over(w_monthly) * 100,
            4
        )
    )
    .withColumn(
        "export_momentum",
        F.when(col("export_change_pct") <= -10.0, F.lit(1)).otherwise(F.lit(0))
    )
    .na.fill(0.0, subset=["export_change_pct"])
)

# ── 캘린더 뼈대에 LEFT JOIN ──────────────────────────────────────
semi_range = semi_derived_df.agg(
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date")
).collect()[0]

calendar_base = (
    kr_biz_days
    .filter(
        (col("date") >= semi_range["min_date"]) &
        (col("date") <= semi_range["max_date"])
    )
)

semi_joined_df = calendar_base.join(semi_derived_df, on="date", how="left")

# ── Forward Fill (모든 컬럼) ─────────────────────────────────────
w_ffill = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)

semi_ffill_df = semi_joined_df
for c in ["export_usd", "export_change_pct", "export_momentum"]:
    semi_ffill_df = semi_ffill_df.withColumn(
        c, F.last(col(c), ignorenulls=True).over(w_ffill)
    )

print(f"✅ Forward Fill 완료: {semi_ffill_df.count()}일 (영업일 기준)")
print(f"   null 잔존: {semi_ffill_df.filter(col('export_usd').isNull()).count()}건")
display(
    semi_ffill_df
    .select("date", "export_usd", "export_change_pct", "export_momentum")
    .orderBy("date")
    .limit(15)
)

# 5. 파생 컬럼 생성

| 컬럼명 | 계산 방법 | 의미 |
|---|---|---|
| `export_change_pct` | (오늘 - 이전발표) / 이전발표 × 100 | 전기 대비 수출 변화율 |
| `export_momentum` | export_change_pct ≤ -10% 이면 1 | 수출 급감 경보 플래그 |

### 쉬운 설명

**export_momentum (수출 급감 경보)**
> 반도체 수출이 10일 전보다 10% 이상 줄었다면
> 글로벌 반도체 수요가 꺾이기 시작했다는 신호입니다.
> 이 플래그가 켜지는 날은 삼성·하이닉스 주가 하락을 예고합니다.


In [0]:
# Cell 11에서 파생 컬럼이 이미 계산+Forward Fill 됨
# 여기서는 파티션 컬럼만 추가

semi_featured_df = (
    semi_ffill_df
    .withColumn("year",  F.year("date"))
    .withColumn("month", F.month("date"))
)

print("✅ 파티션 컬럼 추가 완료")
display(
    semi_featured_df
    .select("date", "export_usd", "export_change_pct", "export_momentum")
    .orderBy("date")
    .limit(15)
)

# 6. 데이터 검증


In [0]:
date_range = semi_featured_df.agg(
    F.min("date").alias("start_date"),
    F.max("date").alias("end_date"),
    F.count("date").alias("영업일수")
).collect()[0]

print("=== 📅 날짜 범위 ===")
print(f"  시작: {date_range['start_date']}")
print(f"  종료: {date_range['end_date']}")
print(f"  영업일수: {date_range['영업일수']}일")

total = semi_featured_df.count()
for c in ["export_usd", "export_change_pct", "export_momentum"]:
    null_cnt = semi_featured_df.filter(col(c).isNull()).count()
    status = "✅" if null_cnt <= 1 else "⚠️"
    print(f"  {status} {c}: null {null_cnt}건")

momentum_cnt = semi_featured_df.filter(col("export_momentum") == 1).count()
print(f"\n=== 🚨 수출 급감 경보 발생: {momentum_cnt}건 ===")
display(
    semi_featured_df.filter(col("export_momentum") == 1)
    .select("date", "export_usd", "export_change_pct")
    .orderBy("date")
)


# 7. Silver 저장


In [0]:
(
    semi_featured_df
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(SEMI_SILVER_PATH)
)

saved_files = dbutils.fs.ls(SEMI_SILVER_PATH)
print(f"✅ Silver 저장 완료")
print(f"   경로     : {SEMI_SILVER_PATH}")
print(f"   파티션 수: {len(saved_files)}개")


# 8. 저장 확인 (sanity check)


In [0]:
df_check = spark.read.parquet(SEMI_SILVER_PATH)
print(f"✅ silver 읽기 확인 — 행수: {df_check.count():,}")
print(f"   컬럼: {df_check.columns}")
display(df_check.orderBy("date").limit(10))

   
## curated 컨테이너 Silver 레이어 전체 정리

### 요약
| # | Silver | 노트북 | 행수 | 컬럼수 | 주기 | 날짜기준 |
|---|---|---|---|---|---|---|
| 1 | yfinance | 01_yfinance_raw_to_silver | 2,488 | 15 | 일별 | `effective_kr_date` (한국 유효 영업일) |
| 2 | fred | 02_fred_raw_to_silver | 250 | 12 | 일별 | `date` (미국 영업일) |
| 3 | fx | 03_fx_raw_to_silver | 243 | 10 | 일별 | `date` (한국 영업일) |
| 4 | semiconductor | 04_semiconductor_raw_to_silver | 511 | 6 | 일별 (FF) | `date` (한국 영업일) |
| 5 | kfinance | 05_kfinance_raw_to_silver | 28 | 9 | 월별 | `date` (한국 영업일) |

---

### 1. yfinance (2,488행 · 15컬럼)
| 컬럼 | 타입 | 설명 |
|---|---|---|
| `date` | DateType | 원래 거래일 |
| `Ticker` | StringType | 종목코드 (005930.KS, NVDA 등 10종목) |
| `Name` | StringType | 종목명 |
| `Open` / `High` / `Low` / `Close` | DoubleType | 시가·고가·저가·종가 |
| `Volume` | DoubleType | 거래량 |
| `effective_kr_date` | DateType | 한국 유효 영업일 (미국 +1영업일 시프트) |
| `log_return` | DoubleType | 로그 수익률, 첫 행 0 |
| `volatility_gk` | DoubleType | Garman-Klass 일별 변동성 |
| `volatility_5d` | DoubleType | 5일 실현 변동성, 첫 행 0 |
| `year` / `month` / `day` | IntegerType | 파티션 컬럼 |

### 2. fred (250행 · 12컬럼)
| 컬럼 | 타입 | 설명 |
|---|---|---|
| `date` | DateType | 미국 영업일 |
| `DGS10` | DoubleType | 10년물 국채금리 |
| `DGS2` | DoubleType | 2년물 국채금리 |
| `T10Y2Y` | DoubleType | 10년-2년 금리차 (FRED 원본) |
| `DFF` | DoubleType | 연방기금금리 (FFR) |
| `DFII10` | DoubleType | 10년물 실질금리 (TIPS) |
| `BAMLH0A0HYM2` | DoubleType | 하이일드 스프레드 (신용위험) |
| `yield_spread` | DoubleType | DGS10 - DGS2 (직접 계산) |
| `yield_spread_change` | DoubleType | 금리차 일별 변화량, 첫 행 0 |
| `stagnation_pressure` | DoubleType | DGS10 + DFII10 (스태그플레이션 압력) |
| `year` / `month` | IntegerType | 파티션 컬럼 |

### 3. fx (243행 · 10컬럼)
| 컬럼 | 타입 | 설명 |
|---|---|---|
| `date` | DateType | 한국 영업일 |
| `usd_krw` | DoubleType | 일별 환율 |
| `usd_krw_low` / `usd_krw_high` | DoubleType | 일중 최저·최고 |
| `수집횟수` | LongType | 하루 수집 건수 |
| `usd_krw_change` | DoubleType | 전일 대비 변화량, 첫 행 0 |
| `usd_krw_pct` | DoubleType | 전일 대비 변화율 (%), 첫 행 0 |
| `risk_off_flag` | IntegerType | 달러 급등 신호 (≥1% 상승 시 1) |
| `year` / `month` | IntegerType | 파티션 컬럼 |

### 4. semiconductor (511행 · 6컬럼) ← 현재 노트북
| 컬럼 | 타입 | 설명 |
|---|---|---|
| `date` | DateType | 한국 영업일 (Forward Fill 완료) |
| `export_usd` | DoubleType | 월별 반도체 수출액 (USD, HS8542 합산) |
| `export_change_pct` | DoubleType | 전월비 수출 변화율 (%), 첫 행 0 |
| `export_momentum` | IntegerType | 수출 급감 경보 (≤-10% 시 1) |
| `year` / `month` | IntegerType | 파티션 컬럼 |

### 5. kfinance (28행 · 9컬럼)
| 컬럼 | 타입 | 설명 |
|---|---|---|
| `date` | DateType | 기준일자 (해당 월 마지막 영업일) |
| `call_volume` | DoubleType | 코스피200 콜 거래량 합계 |
| `call_oi` | DoubleType | 코스피200 콜 미결제약정 합계 |
| `avg_iv` | DoubleType | 거래량 가중평균 내재변동성 (VIX 대용) |
| `iv_change` | DoubleType | IV 전월비 변화량, 첫 행 0 |
| `vol_change_pct` | DoubleType | 거래량 전월비 변화율 (%), 첫 행 0 |
| `iv_surge_flag` | IntegerType | IV 급등 신호 (전월비 +5 이상 시 1) |
| `year` / `month` | IntegerType | 파티션 컬럼 |

---

### Gold JOIN 시 참고

```
yf_close_pivot (effective_kr_date 기준)
    LEFT JOIN fred_silver       ON date       ← 일별
    LEFT JOIN fx_silver         ON date       ← 일별
    LEFT JOIN semi_silver       ON date       ← 일별 (FF완료)
    LEFT JOIN kfin_silver       ON date       ← 월별 → Forward Fill 필요
```

> ⚠️ **FX 주의**: FX `date`(한국 영업일) ≠ yfinance `effective_kr_date`(시프트) → JOIN 매칭율 낮음
> ⚠️ **kfinance 주의**: 월별 28행만 존재 → Gold에서 Forward Fill로 일별 확장 필요